# Benchmark de descarga Rigol MSO2102A
Compara los caminos `legacy` y `fast` usando el script canónico `benchmark_oscrigol_download.py`.


## 1. Parámetros de prueba
Define el hardware, el transporte, el backend VISA y la señal esperada antes de medir.


In [ ]:
RUN_HARDWARE = False  # Cambiar a True para medir con el Rigol real.
IP_ADDRESS = "192.168.2.2"
TRANSPORT = "socket"  # "socket" recomendado; "instr" usa VXI-11/INSTR.
VISA_BACKEND = "pyvisa"  # "pyvisa" por defecto; "nivisa" usa NI-VISA si está instalado.
SOCKET_PORT = 5555

ITERATIONS = 10
MDEPTH = 70000
CHANNELS = (1,)
BANDWIDTH = "20M"
COUPLING = "AC"
IMPEDANCE = "OMEG"
TRIGGER_SOURCE = "CHAN1"
TRIGGER_COUPLING = "AC"
TRIGGER_LEVEL = 100e-3
TRIGGER_SLOPE = "POS"
VSCALE = 50e-3
HSCALE = 10e-6

POLL_INTERVAL = 0.005
TRIGGER_TIMEOUT = 5.0
WAVEFORM_DELAY = 0.01
WAVEFORM_POINTS_TIMEOUT = 0.5
ARM_DELAY = 0.05
MAX_RETRIES = 2
MIN_VPP = None
SIGNAL_VPP_THRESHOLD = 0.05
METHODS = "legacy,fast"


## 2. Carga del benchmark
Importa el módulo de benchmark para reutilizar sus funciones y evitar duplicar lógica.


In [ ]:
from pathlib import Path
from types import SimpleNamespace
import importlib.util
import csv
import statistics
import time

import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd()
if not (HERE / "benchmark_oscrigol_download.py").exists():
    HERE = Path("oscilloscopes/rigol-MSO2102A").resolve()

spec = importlib.util.spec_from_file_location("benchmark_oscrigol_download", HERE / "benchmark_oscrigol_download.py")
bench = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bench)
print("Benchmark module:", HERE / "benchmark_oscrigol_download.py")


## 3. Argumentos compartidos
Construye un objeto de argumentos compatible con el script de línea de comandos.


In [ ]:
def make_args(transport=TRANSPORT, visa_backend=VISA_BACKEND, iterations=ITERATIONS, methods=METHODS):
    args = SimpleNamespace(
        backend="hardware" if RUN_HARDWARE else "fake",
        transport=transport,
        ip=IP_ADDRESS,
        socket_port=SOCKET_PORT,
        visa_backend=visa_backend,
        iterations=iterations,
        mdepth=MDEPTH,
        channels=CHANNELS,
        coupling=bench.expand_per_channel(COUPLING, CHANNELS),
        impedance=bench.expand_per_channel(IMPEDANCE, CHANNELS),
        bandwidth=BANDWIDTH,
        trigger_source=TRIGGER_SOURCE,
        trigger_coupling=TRIGGER_COUPLING,
        trigger_level=TRIGGER_LEVEL,
        trigger_slope=TRIGGER_SLOPE,
        vscale=VSCALE,
        hscale=HSCALE,
        poll_interval=POLL_INTERVAL,
        trigger_timeout=TRIGGER_TIMEOUT,
        waveform_delay=WAVEFORM_DELAY,
        waveform_points_timeout=WAVEFORM_POINTS_TIMEOUT,
        arm_delay=ARM_DELAY,
        max_retries=MAX_RETRIES,
        min_vpp=MIN_VPP,
        signal_vpp_threshold=SIGNAL_VPP_THRESHOLD,
        methods=bench.parse_methods(methods),
        output=str(HERE / f"benchmark_oscrigol_{transport}_{visa_backend}_ch1.csv"),
    )
    return args

args = make_args()
args


## 4. Prueba rápida de conexión
Abre comunicación, consulta `*IDN?` y cierra para verificar que el equipo responde.


In [ ]:
if RUN_HARDWARE:
    scope = bench.build_scope(args, mode="fast")
    scope.initComm()
    try:
        print(scope.getID())
    finally:
        scope.closeComm()
else:
    print("RUN_HARDWARE=False; se omite la conexión real.")


## 5. Una adquisición por método
Captura una señal con `legacy` y otra con `fast` para verificar forma de onda antes del benchmark.


In [ ]:
def capture_once(args, mode, method_name):
    scope = bench.build_scope(args, mode=mode)
    if args.backend == "fake":
        scope._osci = bench.FakeRigolResource(mdepth=args.mdepth)
    else:
        bench.prepare_hardware_scope(scope, args)
    try:
        method = getattr(scope, method_name)
        start = time.perf_counter()
        if method_name == "getchannelsFast":
            values, vmax = method(
                args.channels,
                args.mdepth,
                poll_interval=args.poll_interval,
                trigger_timeout=args.trigger_timeout,
                waveform_delay=args.waveform_delay,
                arm_delay=args.arm_delay,
                max_retries=args.max_retries,
                min_vpp=args.min_vpp,
                waveform_points_timeout=args.waveform_points_timeout,
            )
        else:
            values, vmax = method(args.channels, args.mdepth)
        elapsed = time.perf_counter() - start
        return {"mode": mode, "method": method_name, "values": np.asarray(values), "vmax": vmax, "elapsed_s": elapsed}
    finally:
        if args.backend == "hardware":
            scope.closeComm()

single_results = []
for mode, method_name in args.methods:
    result = capture_once(args, mode, method_name)
    single_results.append(result)
    print(f"{method_name}: {result['elapsed_s']:.3f}s, samples={result['values'].shape[-1]}, vmax_ch2={result['vmax']}")


## 6. Visualización de prueba
Grafica las capturas únicas para confirmar que no se está midiendo ruido o una señal falsa.


In [ ]:
if single_results:
    t_us = np.linspace(-7 * HSCALE, 7 * HSCALE, single_results[0]["values"].shape[-1]) * 1e6
    fig, axes = plt.subplots(len(single_results), 1, figsize=(9, 3 * len(single_results)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, result in zip(axes, single_results):
        y = result["values"]
        if y.ndim > 1:
            y = y[0]
        ax.plot(t_us, y * 1e3)
        ax.set_title(f"{result['method']} - {result['elapsed_s']:.3f}s")
        ax.set_ylabel("mV")
        ax.grid(True, linestyle="--", alpha=0.5)
    axes[-1].set_xlabel("tiempo [us]")
else:
    print("No hay adquisiciones de prueba.")


## 7. Benchmark principal
Ejecuta las iteraciones configuradas y reutiliza `benchmark_method()` del script canónico.


In [ ]:
def run_methods(args):
    rows = []
    for mode, method_name in args.methods:
        scope = bench.build_scope(args, mode=mode)
        if args.backend == "fake":
            scope._osci = bench.FakeRigolResource(mdepth=args.mdepth)
        else:
            bench.prepare_hardware_scope(scope, args)
        try:
            method_rows = bench.benchmark_method(scope, method_name, args)
            rows.extend(method_rows)
            stats = bench.summarize(method_rows)
            print(
                f"{args.transport}/{args.visa_backend} {method_name}: "
                f"n={stats['n']} failures={stats['failures']} "
                f"median={stats['median_s']:.3f}s mean={stats['mean_s']:.3f}s "
                f"min={stats['min_s']:.3f}s max={stats['max_s']:.3f}s"
            )
        finally:
            if args.backend == "hardware":
                scope.closeComm()
    return rows

benchmark_rows = run_methods(args)
benchmark_rows[:3]


## 8. Calidad de señal
Cuenta capturas válidas, errores, descargas parciales y señales por debajo del umbral de Vpp.


In [ ]:
def print_quality(rows):
    groups = {}
    for row in rows:
        groups.setdefault(row["method"], []).append(row)
    for method, group in groups.items():
        valid = sum(row["samples_ok"] and row["signal_ok"] and not row["error"] for row in group)
        errors = sum(bool(row["error"]) for row in group)
        bad_samples = sum(not row["samples_ok"] for row in group)
        bad_signal = sum(row["samples_ok"] and not row["signal_ok"] for row in group)
        retries = sum(row["retries"] for row in group)
        print(f"{method}: validas={valid}/{len(group)}, errores={errors}, samples_mal={bad_samples}, senal_baja={bad_signal}, retries={retries}")

print_quality(benchmark_rows)


## 9. Evolución temporal
Grafica el tiempo por iteración y permite ver retries o colas lentas.


In [ ]:
if benchmark_rows:
    fig, ax = plt.subplots(figsize=(9, 4))
    for method in sorted({row["method"] for row in benchmark_rows}):
        rows = [row for row in benchmark_rows if row["method"] == method]
        ax.plot([row["iteration"] for row in rows], [row["elapsed_s"] for row in rows], "o-", label=method)
    ax.set_xlabel("iteración")
    ax.set_ylabel("tiempo [s]")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend()
else:
    print("No hay datos temporales.")


## 10. Guardado CSV
Guarda los resultados con el mismo formato que el script de benchmark.


In [ ]:
output_path = Path(args.output)
bench.write_csv(output_path, benchmark_rows)
print("CSV escrito en", output_path)


## 11. Comparación socket/backend
Ejecuta tandas equivalentes para comparar `socket+pyvisa` contra `socket+nivisa`.


In [ ]:
RUN_COMPARISON = False
COMPARISON = [
    ("socket", "pyvisa"),
    ("socket", "nivisa"),
]

comparison_rows = []
if RUN_COMPARISON:
    for transport, visa_backend in COMPARISON:
        comp_args = make_args(transport=transport, visa_backend=visa_backend, iterations=ITERATIONS, methods="fast")
        rows = run_methods(comp_args)
        for row in rows:
            row["transport"] = transport
            row["visa_backend"] = visa_backend
        comparison_rows.extend(rows)
    comparison_path = HERE / "benchmark_oscrigol_socket_backend_comparison_ch1.csv"
    bench.write_csv(comparison_path, comparison_rows)
    print("CSV comparación:", comparison_path)
else:
    print("RUN_COMPARISON=False; se omite comparación múltiple.")


## Interpretación esperada
`socket+fast` debería ser el camino más rápido, y `samples_ok=True` debe cumplirse siempre para considerar válida una descarga.
